In [1]:
import pandas as pd
import numpy as np
import ydata_profiling
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from copy import deepcopy
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error
import numpy as np


In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print("✅ Train shape:", train.shape)
if test is not None:
    print("✅ Test shape:", test.shape)

print("\nTrain info:")
print(train.info())


✅ Train shape: (517754, 14)
✅ Test shape: (172585, 13)

Train info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accide

In [4]:
num_features = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
cat_features = ['road_type', 'lighting', 'weather', 'time_of_day']
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']

print("✅ Numeric features:", num_features)
print("✅ Categorical features:", cat_features)
print("✅ Boolean features:", bool_features)


✅ Numeric features: ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
✅ Categorical features: ['road_type', 'lighting', 'weather', 'time_of_day']
✅ Boolean features: ['road_signs_present', 'public_road', 'holiday', 'school_season']


In [5]:
for col in bool_features:
    train[col] = train[col].astype(int)
    if test is not None:
        test[col] = test[col].astype(int)

ordinal_features = ['time_of_day', 'lighting']
onehot_features = ['weather', 'road_type']  

ordinal_categories = [
    ['morning', 'afternoon', 'evening'],  # time_of_day
    ['daylight', 'dim', 'night']  # lighting
]


ordinal_transformer = OrdinalEncoder(categories=ordinal_categories)
onehot_transformer = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


In [6]:
TARGET = "accident_risk"

# Interaction features
def add_interactions(df):
    df = deepcopy(df)
    df['curv_x_spd'] = df['curvature'] * df['speed_limit']
    df['lanes_x_spd'] = df['num_lanes'] * df['speed_limit']
    return df

train = add_interactions(train)
test = add_interactions(test)

# Update numeric features list to include new interactions
interaction_features = ['curv_x_spd', 'lanes_x_spd']
num_features_extended = num_features + interaction_features


preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features_extended),
        ('ord', ordinal_transformer, ordinal_features),
        ('ohe', onehot_transformer, onehot_features),
        ('bool', 'passthrough', bool_features),
    ],
    remainder='drop'
)

X_train = train.drop(columns=[TARGET, 'id'])
X_test = test.drop(columns=['id'])
y_train = train[TARGET]

preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("✅ Feature engineering and preprocessing complete")
print("Transformed train shape:", X_train_transformed.shape)
print("Transformed test shape:", X_test_transformed.shape)


✅ Feature engineering and preprocessing complete
Transformed train shape: (517754, 18)
Transformed test shape: (172585, 18)


In [ ]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',  
    n_jobs=-1,                     
    random_state=42
)

param_distributions = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [1, 1.5, 2, 3]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=100,            
    scoring='neg_root_mean_squared_error',  
    cv=5,                 
    verbose=2,
    random_state=42
)

random_search.fit(X_train_transformed, y_train)


# Best model and train RMSE
print("Best parameters found:", random_search.best_params_)
best_model = random_search.best_estimator_
y_pred_train = best_model.predict(X_train_transformed)
rmse_train = root_mean_squared_error(y_train, y_pred_train)
print("Train RMSE:", rmse_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, n_estimators=200, reg_alpha=0.01, reg_lambda=3, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, n_estimators=200, reg_alpha=0.01, reg_lambda=3, subsample=0.8; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, n_estimators=200, reg_alpha=0.01, reg_lambda=3, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, n_estimators=200, reg_alpha=0.01, reg_lambda=3, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.2, max_depth=7, n_estimators=200, reg_alpha=0.01, reg_lambda=3, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=9, n_estimators=300, reg_alpha=1, reg_lambda=2, subsample=1.0; total time=   2.0s
[CV] END colsample_bytree=0.6, learning_rate=0.01, max_depth=9, n_est

In [ ]:
y_test_pred = best_model.predict(X_test_transformed)

submission = pd.DataFrame({
    'id': test['id'],
    'accident_risk': y_test_pred
})

submission['accident_risk'] = submission['accident_risk'].astype(float)

submission.to_csv('XGBoost.csv', index=False)
print("✅ Submission file created: XGBoost.csv")


✅ Submission file created: XGBoost.csv
